# <span style="color:purple"> Apply Filters to Data </span>

## <span style="color:purple"> Import Data </span>

In [ ]:
from scipy import signal
import pandas as pd

df = pd.read_parquet("../data/proc/01_imported.parquet")

In [ ]:
def lowpass_filter(series, cutoff_period, sampling_interval, order=2):
    """
    series:            pandas Series of values (no NaNs)
    cutoff_period:     period to filter at, in same units as sampling_interval
                       e.g. 24 hours — anything faster gets removed
    sampling_interval: time between samples, i.e. 0.5 for 30-minute data in hours for AMISOR MicroCATs
    order:             filter order
    """
    # Calculate the sampling frequency of the filter using the provided time between sampling
    fs = 1 / sampling_interval
    # Calculate the d
    cutoff_freq = 1 / cutoff_period        # cutoff frequency
    sos = butter(order, cutoff_freq, btype='low', fs=fs, output='sos')
    return sosfiltfilt(sos, series)

In [ ]:
for (bh, cat), group in df.groupby(["borehole", "cat"]):
    mask = (df["borehole"] == bh) & (df["cat"] == cat)
    df.loc[mask, "in_situ_temperature_smooth"] = lowpass_filter(
        group["temperature"].values,
        cutoff_period=24,       # e.g. remove anything faster than 24 hours
        sampling_interval=0.5   # e.g. data every 30 minutes, in hours
    )